# Motion-Guided Self-Supervised Learning for Cardiac Cine MRI
## Notebook 08: Ablation Study, Robustness Analysis & Results Aggregation

This notebook provides the consolidated evaluation, comparison, ablation, and robustness infrastructure for the research framework.

### 1. Ablation Study (5 Major Variants)
- **Variant A (Supervised Baseline)**: Standard 2D U-Net initialized from scratch with random weights
- **Variant B (SSL Only)**: 2D U-Net with weights initialized from temporal masked autoencoding (`SharedEncoder`)
- **Variant C (SSL + Motion)**: SSL pretraining + unsupervised motion estimation & differentiable temporal warping regularization
- **Variant D (SSL + Pseudo-Labels)**: SSL pretraining + confidence-filtered pseudo-labeling on unlabeled frames (no motion)
- **Variant E (Full Proposed Pipeline)**: Full framework integrating SSL + Motion Regularization + Confidence-Filtered Pseudo-Labels

### 2. Label-Efficiency Regimes
- **10% Labels**: 7 patients (128 slices) - extreme annotation scarcity
- **25% Labels**: 17 patients (318 slices) - low annotation regime
- **50% Labels**: 35 patients (660 slices) - moderate annotation regime
- **100% Labels**: 70 patients (1,324 slices) - full annotation upper bound

### 3. Patient-Level & Temporal Metrics
- **Per-Patient Aggregation**: First averages slices per patient, then computes mean $\pm$ std across test patients (avoids slice-count bias)
- **Class Metrics**: LV cavity Dice, Myocardium Dice, RV cavity Dice, Mean Foreground Dice, HD95 (mm)
- **Temporal Consistency**: Frame-to-frame uncompensated vs. motion-warped segmentation agreement and gain
- **Pseudo-Label Quality**: Acceptance rate ($\tau \in [0.6, 0.95]$), accuracy on accepted pixels, and foreground Dice

### 4. Robustness Variations
- **Random Seeds**: Seeds 42, 123, 456
- **Noise Perturbations**: Additive Gaussian noise ($\sigma \in [0.05, 0.10]$)
- **Intensity Variations**: Multiplicative scaling ($0.9\times, 1.1\times$)
- **Temporal Interval Variations**: Consecutive frame spacing ($\Delta t = 1$ vs $\Delta t = 2$)

---
> **COMPUTE CONSTRAINT NOTICE**:
> This local development system is used strictly for pipeline implementation, sanity validation, and table/figure generation.
> Actual multi-epoch neural network training and GPU inference are executed on the separate GPU training system (`TRAINING MACHINE ONLY`).
> When result files from the training system are synced to `results/`, this notebook consumes them directly to generate publication tables and figures.

In [ ]:
# =============================================================================
# Cell 1: Environment Setup and Path Verification
# =============================================================================
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

print(f"Project root resolved: {project_root}")
results_dir = project_root / "results"
tables_dir = results_dir / "tables"
figures_dir = results_dir / "figures"
exp_dir = results_dir / "experiments"
print(f"Results directory: {results_dir}")

In [ ]:
# =============================================================================
# Cell 2: Check Experiment Registry & Display Training System Notice
# =============================================================================
registry_path = exp_dir / "experiment_registry.json"

if registry_path.exists():
    with open(registry_path, "r", encoding="utf-8") as f:
        registry_data = json.load(f)
    print(f"Loaded experiment registry: {len(registry_data)} run records found.")
    for exp_id, rec in registry_data.items():
        status = rec.get('status', 'unknown')
        mode = rec.get('mode', 'N/A')
        frac = rec.get('label_fraction', 'N/A')
        print(f"  - [{exp_id}]: mode={mode}, fraction={frac}%, status={status}")
else:
    registry_data = {}
    print("Registry file not yet found. Initializing empty registry.")

# Check if full GPU experimental runs have been completed
has_trained_experiments = any(
    rec.get("status") == "completed_gpu" for rec in registry_data.values()
)

if not has_trained_experiments:
    display(Markdown(
        "> **NOTICE: FULL EXPERIMENTS PENDING GPU EXECUTION**\n"
        "> \n"
        "> Full training of the 16 experiment matrix and 5 ablation variants must first be run on the separate GPU training system.\n"
        "> \n"
        "> **Commands to execute on Training System**:\n"
        "> ```bash\n"
        "> # 1. Train full proposed pipeline with 10% labels\n"
        "> python src/experiment_runner.py --mode full_pipeline --label-fraction 10 --device cuda\n"
        "> \n"
        "> # 2. Run ablation matrix (10% labels)\n"
        "> for mode in supervised ssl_finetune ssl_motion ssl_pseudo full_pipeline; do\n"
        ">     python src/experiment_runner.py --mode $mode --label-fraction 10 --device cuda\n"
        "> done\n"
        "> \n"
        "> # 3. Re-aggregate tables and figures\n"
        "> python src/aggregate_results.py --generate-plots\n"
        "> ```"
    ))
else:
    display(Markdown("**SUCCESS**: Completed GPU experimental runs detected in registry."))

In [ ]:
# =============================================================================
# Cell 3: Ablation Study Comparison Table (5 Major Variants)
# =============================================================================
ablation_csv = tables_dir / "ablation_table.csv"
ablation_md = tables_dir / "ablation_table.md"

print(f"Loading Ablation Table from: {ablation_csv}")
if ablation_csv.exists():
    df_ablation = pd.read_csv(ablation_csv)
    display(Markdown("### Table 1: Ablation Study Comparison Across 5 Pipeline Components"))
    display(df_ablation.fillna("Pending GPU Run"))
else:
    print("Ablation table CSV not found. Run 'python src/aggregate_results.py' to generate.")

In [ ]:
# =============================================================================
# Cell 4: Label-Efficiency Comparison Table (10%, 25%, 50%, 100% Regimes)
# =============================================================================
le_csv = tables_dir / "label_efficiency_table.csv"

print(f"Loading Label-Efficiency Table from: {le_csv}")
if le_csv.exists():
    df_le = pd.read_csv(le_csv)
    display(Markdown("### Table 2: Label-Efficiency Matrix Across Annotation Regimes"))
    display(df_le.fillna("Pending GPU Run"))
else:
    print("Label-efficiency table CSV not found.")

In [ ]:
# =============================================================================
# Cell 5: Patient-Level Test Evaluation Table (20 Independent Test Patients)
# =============================================================================
# Note: Slices are aggregated per patient first, then reported per patient
# to prevent slice-count bias from distorting reported metrics.
pat_csv = tables_dir / "patient_metrics_table.csv"

print(f"Loading Patient Metrics Table from: {pat_csv}")
if pat_csv.exists():
    df_pat = pd.read_csv(pat_csv)
    display(Markdown("### Table 3: Patient-Level Performance Breakdown (Test Cohort, N=20)"))
    display(df_pat.head(10).fillna("Pending GPU Run"))
    print(f"Total test patients listed: {len(df_pat)} (Displaying top 10)")
else:
    print("Patient metrics table CSV not found.")

In [ ]:
# =============================================================================
# Cell 6: Robustness Evaluation Table (Seeds, Noise, Scaling, Intervals)
# =============================================================================
rob_csv = tables_dir / "robustness_table.csv"

print(f"Loading Robustness Table from: {rob_csv}")
if rob_csv.exists():
    df_rob = pd.read_csv(rob_csv)
    display(Markdown("### Table 4: Robustness & Stability Under Controlled Variations"))
    display(df_rob.fillna("Pending GPU Run"))
else:
    print("Robustness table CSV not found.")

In [ ]:
# =============================================================================
# Cell 7: Publication-Quality Visualization Figures
# =============================================================================
fig_files = [
    ("label_efficiency_curves.png", "Figure 1: Label-Efficiency Curves across Annotation Regimes"),
    ("ablation_comparison.png", "Figure 2: Component Ablation Study (10% Labels)"),
    ("classwise_performance.png", "Figure 3: Class-Wise Performance (LV vs MYO vs RV)"),
    ("temporal_consistency_plot.png", "Figure 4: Cine Frame-to-Frame Temporal Consistency"),
    ("pseudo_label_quality.png", "Figure 5: Pseudo-Label Confidence Calibration Trade-Off"),
    ("robustness_comparison.png", "Figure 6: Robustness Stability across Perturbations"),
]

for fname, title in fig_files:
    fpath = figures_dir / fname
    display(Markdown(f"#### {title}"))
    if fpath.exists():
        display(Image(filename=str(fpath), width=750))
    else:
        print(f"Figure not found: {fpath}. Run 'python src/aggregate_results.py --generate-plots'")

In [ ]:
# =============================================================================
# Cell 8: Dynamic Re-Aggregation Utility
# =============================================================================
# Run this cell to refresh all tables and plots after syncing new GPU runs
from src.aggregate_results import generate_ablation_table, generate_label_efficiency_table, generate_robustness_table, generate_patient_metrics_table, generate_all_plots

print("Refreshing all comparison tables and figures from current registry...")
generate_ablation_table(registry_data, tables_dir)
generate_label_efficiency_table(registry_data, tables_dir)
generate_robustness_table(registry_data, tables_dir)
generate_patient_metrics_table(None, tables_dir)
generate_all_plots(figures_dir)
print("Re-aggregation completed successfully.")